In [1]:
import sys
import random
import numpy as np
import os
from PIL import Image
from core.MyEnv import MyEnv
from lerobot.datasets.lerobot_dataset import LeRobotDataset

/Users/ningyu/code_before_paper/MyI10Tele/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# If you want to randomize the object positions, set this to None
# If you fix the seed, the object positions will be the same every time
SEED = 0 
# SEED = None <- Uncomment this line to randomize the object positions

REPO_NAME = 'auboI10'
NUM_DEMO = 1 # Number of demonstrations to collect
ROOT = "/Users/ningyu/code_before_paper/MyI10Tele/data" # The root directory to save the demonstrations

In [3]:
I10_path = '/Users/ningyu/code_before_paper/MyI10Tele/assets/aubo_i10_2/aubo_i10.xml'
import mujoco 
model = mujoco.MjModel.from_xml_path(I10_path)
print(model.body_pos)

[[ 0.      0.      0.    ]
 [ 0.      0.      0.    ]
 [ 0.      0.      0.1632]
 [ 0.      0.2013  0.    ]
 [ 0.647   0.      0.    ]
 [ 0.6005  0.      0.    ]
 [ 0.      0.1025  0.    ]
 [ 0.     -0.094   0.    ]]


In [4]:
TASK_NAME = 'Put cube on the black platform' 
xml_path = '/Users/ningyu/code_before_paper/MyI10Tele/assets/aubo_i10_inspire/myscene.xml'
# xml_path = './asset/example_scene_y_i10.xml'
# Define the environment
PnPEnv = MyEnv(xml_path, seed = SEED, state_type = 'joint_angle')


-----------------------------------------------------------------------------
name:[myenv] dt:[0.002] HZ:[500]
 n_qpos:[17] n_qvel:[16] n_qacc:[16] n_ctrl:[7]
 integrator:[IMPLICITFAST]

n_body:[19]
 [0/19] [world] mass:[0.00]kg
 [1/19] [base_link] mass:[2.59]kg
 [2/19] [shoulder_Link] mass:[10.18]kg
 [3/19] [upperArm_Link] mass:[18.10]kg
 [4/19] [foreArm_Link] mass:[4.45]kg
 [5/19] [wrist1_Link] mass:[1.79]kg
 [6/19] [wrist2_Link] mass:[1.63]kg
 [7/19] [wrist3_Link] mass:[0.20]kg
 [8/19] [i10_inspire_flange_link] mass:[0.05]kg
 [9/19] [gripper_base_link] mass:[0.08]kg
 [10/19] [wrist_cam_bracket] mass:[0.00]kg
 [11/19] [gripper_Link1] mass:[0.00]kg
 [12/19] [gripper_Link2] mass:[0.00]kg
 [13/19] [gripper_Link3] mass:[0.00]kg
 [14/19] [tcp_link] mass:[0.00]kg
 [15/19] [front_object_table] mass:[1.00]kg
 [16/19] [agentview_bracket] mass:[0.01]kg
 [17/19] [place_target_platform] mass:[0.08]kg
 [18/19] [cube] mass:[0.07]kg
body_total_mass:[40.23]kg

n_geom:[43]
geom_names:['floor', None,

In [5]:
create_new = True
if os.path.exists(ROOT):
    print(f"Directory {ROOT} already exists.")
    ans = input("Do you want to delete it? (y/n) ")
    if ans == 'y':
        import shutil
        shutil.rmtree(ROOT)
    else:
        create_new = False


if create_new:
    dataset = LeRobotDataset.create(
                repo_id=REPO_NAME,
                root = ROOT, 
                robot_type="aubo_i10_inspire",
                fps=20, # 20 frames per second
                features={
                    "observation.image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channels"],
                    },
                    "observation.wrist_image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channel"],
                    },
                    "observation.state": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["state"], # x, y, z, roll, pitch, yaw
                    },
                    "action": {
                        "dtype": "float32",
                        "shape": (7,),
                        "names": ["action"], # 6 joint angles and 1 gripper
                    },
                    "obj_init": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["obj_init"], # just the initial position of the object. Not used in training.
                    },
                },
                image_writer_threads=10,
                image_writer_processes=5,
        )
else:
    print("Load from previous dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)

Directory /Users/ningyu/code_before_paper/MyI10Tele/data already exists.


In [6]:
action = np.zeros(7)
episode_id = 0
record_flag = False # Start recording when the robot starts moving
while PnPEnv.env.is_viewer_alive() and episode_id < NUM_DEMO:
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # check if the episode is done
        done = PnPEnv.check_success()
        if done: 
            # Save the episode data and reset the environment
            dataset.save_episode()
            PnPEnv.reset(seed = SEED)
            episode_id += 1
        # Teleoperate the robot and get delta end-effector pose with gripper
        action, reset  = PnPEnv.teleop_robot()
        if not record_flag and sum(action) != 0:
            record_flag = True
            print("Start recording")
        if reset:
            # Reset the environment and clear the episode buffer
            # This can be done by pressing 'z' key
            PnPEnv.reset(seed=SEED)
            # PnPEnv.reset()
            dataset.clear_episode_buffer()
            record_flag = False
        # Step the environment
        # Get the end-effector pose and images
        ee_pose = PnPEnv.get_ee_pose()
        agent_image,wrist_image = PnPEnv.grab_image()
        # # resize to 256x256
        agent_image = Image.fromarray(agent_image)
        wrist_image = Image.fromarray(wrist_image)
        agent_image = agent_image.resize((256, 256))
        wrist_image = wrist_image.resize((256, 256))
        agent_image = np.array(agent_image)
        wrist_image = np.array(wrist_image)
        joint_q = PnPEnv.step(action)
        if record_flag:
            # Add the frame to the dataset
            dataset.add_frame({
                "observation.image": agent_image,
                "observation.wrist_image": wrist_image,
                "observation.state": ee_pose,
                "action": joint_q,
                "obj_init": PnPEnv.obj_init_pose,
                "task": TASK_NAME,
            })
        PnPEnv.render(teleop=True)

2026-05-09 15:09:35.922 python[97634:695539] error messaging the mach port for IMKCFRunLoopWakeUpReliable
2026-05-09 15:09:42.824 python[97634:695539] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit


Start recording
